# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaineshchaurasiya20/FlyRank_Ml_Assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### The Data Contract: 5 Plain-Words Answers

**Lane:** Refresh / Content Opportunity Scoring (Prioritizing pages for editorial refresh review)

1. **What one row means for our lane:**
   - **Warehouse Fact Table:** In `fact_content_daily_performance`, one row represents **one daily performance record for one pseudonymized content item of a specific client on a single calendar date** (`report_date × client_hash_id × content_hash_id`).
   - **Lane Feature Frame:** For our content refresh decision task, one row represents **one unique content item (`content_hash_id` within `client_hash_id`) evaluated at a discrete decision moment (end of 2026-03-15)**, aggregating pre-decision search and analytics history to prioritize human review.

2. **Which table(s) we will use:**
   - Primary performance log: `fact_content_daily_performance` (partitioned by month, iterating on mid-panel month `month=2026-03`).
   - Dimension tables: `dim_content` (for static content attributes: word count, content type, main intent) and `dim_clients` (for client onboarding timelines, tracking flags, and client-grouped cross-validation splits).

3. **Which time window:**
   - We evaluate on the mid-panel partition `month=2026-03` (`2026-03-01` to `2026-03-31`) with a strict non-overlapping temporal split around the decision moment:
     - **Observation / Feature Window (Pre-decision):** `2026-03-01` to `2026-03-15` (15 days of pre-decision history knowable before review).
     - **Outcome / Target Window (Post-decision):** `2026-03-16` to `2026-03-31` (16 days of future performance used to evaluate page decline or opportunity).

4. **What we predict or rank (label or proxy):**
   - We predict a forward-looking decline proxy: **`is_declining_future`** (binary indicator = 1 if post-decision 16-day impressions drop below 80% of pre-decision 15-day impressions, i.e., `post_gsc_impressions < 0.8 * pre_gsc_impressions`).
   - The model outputs a priority score to rank pages in order of urgency for editorial intervention.

5. **One thing we deliberately exclude:**
   - We deliberately exclude **all retrospective summary trend fields** (`trend_direction`, `trend_pct` from the starter snapshot) and **any post-decision performance metrics** (`post_gsc_impressions`, `post_gsc_clicks`, post-cutoff GA4 sessions) from the feature set.
   - We also exclude raw identifier hashes (`client_hash_id`, `content_hash_id`) as predictive features, reserving them strictly for joins, grouping, and client-held-out validation.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Secure Hugging Face token retrieval (Colab Secrets or environment variable)
# Never paste or print raw secret tokens in public code cells
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter your Hugging Face Read Token: ")

# Initialize DuckDB and register Hugging Face secret
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

def resolve_table_path(rel_path):
    """Resolves local cached parquet file if available, or falls back to direct hf:// URI."""
    try:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(
            repo_id="FlyRank/internship-warehouse",
            filename=rel_path,
            repo_type="dataset",
            token=HF_TOKEN
        )
    except Exception:
        return f"hf://datasets/FlyRank/internship-warehouse/{rel_path}"

# Resolve paths for warehouse tables
CLIENTS_PATH = resolve_table_path("dim_clients.parquet")
CONTENT_PATH = resolve_table_path("dim_content.parquet")
PERF_PATH = resolve_table_path("fact_content_daily_performance/month=2026-03/data_0.parquet")

# Connection verification: count active vs total clients
client_check = con.sql(f"""
    SELECT 
        COUNT(*) AS total_clients, 
        COUNT(*) FILTER (WHERE is_active IS TRUE) AS active_clients,
        COUNT(*) FILTER (WHERE has_gsc_access IS TRUE) AS clients_with_gsc,
        COUNT(*) FILTER (WHERE has_ga4_access IS TRUE) AS clients_with_ga4
    FROM '{CLIENTS_PATH}'
""").df()
print("Connected to Hugging Face warehouse tables successfully.")
display(client_check)

Connected to Hugging Face warehouse tables successfully.


,total_clients,active_clients,clients_with_gsc,clients_with_ga4
0,104,74,67,54


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every field touched in our lane is categorized into exactly one bucket:

| Bucket | Field Name | Source Table | Description & Rationale |
|---|---|---|---|
| **Feature** | `pre_gsc_impressions` | `fact_content_daily_performance` | Sum of Search Console impressions from `2026-03-01` to `2026-03-15`. Knowable prior to decision. |
| **Feature** | `pre_gsc_clicks` | `fact_content_daily_performance` | Sum of Search Console clicks from `2026-03-01` to `2026-03-15`. Knowable prior to decision. |
| **Feature** | `pre_avg_position` | `fact_content_daily_performance` | Impression-weighted average rank (`SUM(gsc_sum_position) / SUM(gsc_impressions)`) up to March 15. |
| **Feature** | `pre_ga4_engaged_sessions` | `fact_content_daily_performance` | Sum of engaged user sessions up to March 15. Knowable prior to decision. |
| **Feature** | `content_word_count` | `dim_content` | Static word count of the published page. Knowable prior to decision. |
| **Label / Proxy** | `is_declining_future` | Computed from post-window | Binary indicator (1 if `post_gsc_impressions < 0.8 * pre_gsc_impressions`, 0 otherwise). Evaluated on `2026-03-16` to `2026-03-31`. |
| **Context** | `client_hash_id` | `dim_clients` / `fact` | Client pseudonym. Used strictly for grouping, joining, and client-held-out validation. Never a feature. |
| **Context** | `content_hash_id` | `dim_content` / `fact` | Content item pseudonym. Used for entity joins and aggregation. Never a feature. |
| **Context** | `report_date` | `fact_content_daily_performance` | Calendar timestamp defining observation vs outcome window boundaries. |
| **Excluded** | `post_gsc_impressions` / `post_gsc_clicks` | Future window fact data | **Why:** Occurs after the decision timestamp (`2026-03-15`). Using them as features causes direct target leakage. |
| **Excluded** | `trend_direction` / `trend_pct` | Starter dataset / post-hoc | **Why:** Retrospective heuristic metrics computed from the outcome window itself; introduces circular definition leakage. |
| **Excluded** | `client_hash_id` & `content_hash_id` (as features) | Identifier columns | **Why:** Memorizes high-cardinality entity IDs rather than generalizable signals, leading to overfit models. |
| **Excluded** | `ga4_data_available = FALSE` uninstrumented rows | `fact_content_daily_performance` | **Why:** GA4 metrics are zero-filled prior to client integration; filtering prevents treating unmeasured tracking as zero engagement. |

In [2]:
# Verify schemas of tables used in our contract
print("--- Schema: fact_content_daily_performance (candidate columns) ---")
perf_schema = con.sql(f"""
    DESCRIBE SELECT 
        report_date, client_hash_id, content_hash_id, 
        gsc_data_available, ga4_data_available, 
        gsc_impressions, gsc_clicks, gsc_avg_position, ga4_engaged_sessions
    FROM '{PERF_PATH}'
""").df()
display(perf_schema[['column_name', 'column_type']])

print("\n--- Schema: dim_content (static metadata columns) ---")
content_schema = con.sql(f"""
    DESCRIBE SELECT 
        client_hash_id, content_hash_id, word_count, content_type, main_intent 
    FROM '{CONTENT_PATH}'
""").df()
display(content_schema[['column_name', 'column_type']])

--- Schema: fact_content_daily_performance (candidate columns) ---


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,gsc_data_available,BOOLEAN
4,ga4_data_available,BOOLEAN
5,gsc_impressions,BIGINT
6,gsc_clicks,BIGINT
7,gsc_avg_position,DOUBLE
8,ga4_engaged_sessions,BIGINT


,column_name,column_type
0,client_hash_id,VARCHAR
1,content_hash_id,VARCHAR
2,word_count,BIGINT
3,content_type,VARCHAR
4,main_intent,VARCHAR


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Part A: Prove Three Facts with Three Small Queries
We test our claims on the mid-panel month partition `month=2026-03`:
1. **Fact 1 — The Grain:** One row in `fact_content_daily_performance` is uniquely identified by `(report_date, client_hash_id, content_hash_id)`. We probe for duplicates with `HAVING count(*) > 1` — zero rows back proves the grain holds.
2. **Fact 2 — Row Count and Date Span:** The exact row count, date span (`min_date` to `max_date`), client count, and content item count in our slice.
3. **Fact 3 — Availability (`IS TRUE`):** Filter with `IS TRUE` on `gsc_data_available` and `ga4_data_available`, showing exactly how many rows survive.

In [3]:
# ====================================================================
# FACT 1: Grain Verification (Duplicate Probe)
# ====================================================================
print("=== FACT 1: Grain Verification (Duplicate Probe) ===")
q_grain = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_count
FROM '{PERF_PATH}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING duplicate_count > 1
LIMIT 5
"""
df_grain = con.sql(q_grain).df()
print(f"Duplicate rows found: {len(df_grain)}")
display(df_grain)
assert len(df_grain) == 0, "Grain check failed: duplicate rows exist!"
print("Verified: The grain strictly holds — exactly 0 duplicate rows.\n")

# ====================================================================
# FACT 2: Slice Row Count and Date Span
# ====================================================================
print("=== FACT 2: Slice Row Count and Date Span ===")
q_span = f"""
SELECT 
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT content_hash_id) AS total_content_items
FROM '{PERF_PATH}'
"""
df_span = con.sql(q_span).df()
display(df_span)
print(f"Verified: Total rows = {df_span.loc[0, 'total_rows']:,}, spanning {df_span.loc[0, 'min_date']} to {df_span.loc[0, 'max_date']}.\n")

# ====================================================================
# FACT 3: Availability Verification with IS TRUE
# ====================================================================
print("=== FACT 3: Availability Verification with IS TRUE ===")
q_avail = f"""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS pct_gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    ROUND(COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS pct_ga4_available,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS both_available_rows,
    ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS pct_both_available
FROM '{PERF_PATH}'
"""
df_avail = con.sql(q_avail).df()
display(df_avail)
print("Verified: Availability filtered with IS TRUE. Exactly 3,611,061 rows (36.69%) have GSC available, and 364,347 rows (3.70%) have both available.")

=== FACT 1: Grain Verification (Duplicate Probe) ===
Duplicate rows found: 0


,report_date,client_hash_id,content_hash_id,duplicate_count


,total_rows,min_date,max_date,total_clients,total_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


,total_rows,gsc_available_rows,pct_gsc_available,ga4_available_rows,pct_ga4_available,both_available_rows,pct_both_available
0,9841378,3611061,36.69,413966,4.21,364347,3.7


### Part B: Five Features, Max (Feature Frame)

We build a small feature frame for our lane from `month=2026-03` aggregated at the content item level across the pre-decision window (`2026-03-01` to `2026-03-15`), joined with static metadata from `dim_content`.

Every feature has a one-line temporal guarantee: **"knowable at the decision moment because..."**:
1. **`pre_gsc_impressions`**: Knowable at the decision moment because Search Console daily impressions up to March 15 have already been logged and synced to BigQuery.
2. **`pre_gsc_clicks`**: Knowable at the decision moment because organic clicks through March 15 have already landed in the daily warehouse partition.
3. **`pre_avg_position`**: Knowable at the decision moment because impression-weighted rank depth measures SERP visibility accumulated prior to the decision point.
4. **`pre_ga4_engaged_sessions`**: Knowable at the decision moment because on-site user engagement through March 15 was recorded in GA4 prior to review.
5. **`content_word_count`**: Knowable at the decision moment because article word count is an existing static property of the published page present in the CMS/database prior to review.

In [4]:
# Build the 5-feature frame from month=2026-03
q_features = f"""
WITH pre_window AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS pre_gsc_impressions,
        SUM(gsc_clicks) AS pre_gsc_clicks,
        CASE WHEN SUM(gsc_impressions) > 0 
             THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
             ELSE 0.0 END AS pre_avg_position,
        SUM(COALESCE(ga4_engaged_sessions, 0)) AS pre_ga4_engaged_sessions
    FROM '{PERF_PATH}'
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 10
),
post_window AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS post_gsc_impressions
    FROM '{PERF_PATH}'
    WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT 
    p.client_hash_id,
    p.content_hash_id,
    p.pre_gsc_impressions,
    p.pre_gsc_clicks,
    p.pre_avg_position,
    p.pre_ga4_engaged_sessions,
    COALESCE(c.word_count, 0) AS content_word_count,
    -- Observed future outcome (label) computed strictly on post-decision window
    COALESCE(post.post_gsc_impressions, 0) AS post_gsc_impressions,
    CASE WHEN COALESCE(post.post_gsc_impressions, 0) < (p.pre_gsc_impressions * 0.8) THEN 1 ELSE 0 END AS is_declining_label
FROM pre_window p
LEFT JOIN post_window post 
    ON p.client_hash_id = post.client_hash_id AND p.content_hash_id = post.content_hash_id
LEFT JOIN '{CONTENT_PATH}' c 
    ON p.client_hash_id = c.client_hash_id AND p.content_hash_id = c.content_hash_id
"""

feature_frame = con.sql(q_features).df()
print(f"Feature frame successfully built: {feature_frame.shape[0]:,} content items × {feature_frame.shape[1]} columns.")
display(feature_frame.head(5))

print("\nClass balance of forward-looking decline label:")
display(feature_frame['is_declining_label'].value_counts(normalize=True).rename('proportion').to_frame())

Feature frame successfully built: 120,513 content items × 9 columns.


,client_hash_id,content_hash_id,pre_gsc_impressions,pre_gsc_clicks,pre_avg_position,pre_ga4_engaged_sessions,content_word_count,post_gsc_impressions,is_declining_label
0,client_73cda7b4e4f265ea,content_53181e97e629da72,938.0,5.0,3.092751,0.0,0,1216.0,0
1,client_73cda7b4e4f265ea,content_9e112029a7e004eb,896.0,4.0,22.255580,0.0,0,890.0,0
2,client_73cda7b4e4f265ea,content_84f28d7f7d286c5c,135.0,0.0,32.962963,0.0,0,122.0,0
3,client_73cda7b4e4f265ea,content_cad5971a1c9c2d0a,1389.0,5.0,6.850972,0.0,2790,1423.0,0
4,client_73cda7b4e4f265ea,content_d87d9e2474306958,1717.0,3.0,3.573675,0.0,0,566.0,1


,proportion
is_declining_label,
0,0.703509
1,0.296491


### Part C: The Trap — Deliberate Leakage Experiment

In notebook 02, we learned how label leakage artificially inflates validation scores. Here we perform that exact experiment on real warehouse data:
1. **Honest Baseline:** We train a classifier using only the 5 honest, pre-decision features.
2. **The Trap:** We deliberately inject a future-outcome column (`post_gsc_impressions`), which occurred *after* the decision moment. Watch the validation score jump artificially toward near-perfect!
3. **The Fix:** We remove the leaky column, verify that only knowable pre-decision features remain, and keep the honest number.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split

honest_features = [
    'pre_gsc_impressions',
    'pre_gsc_clicks',
    'pre_avg_position',
    'pre_ga4_engaged_sessions',
    'content_word_count'
]

X_honest = feature_frame[honest_features].copy()
y = feature_frame['is_declining_label'].copy()

# Stratified train/test split
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)

# ====================================================================
# Step 1: Fit Honest Baseline Model
# ====================================================================
rf_honest = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
rf_honest.fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, rf_honest.predict_proba(X_te)[:, 1])
honest_acc = accuracy_score(y_te, rf_honest.predict(X_te))

print("=== 1. HONEST MODEL (5 knowable pre-decision features) ===")
print(f"Features used: {honest_features}")
print(f"Honest ROC-AUC : {honest_auc:.4f}")
print(f"Honest Accuracy: {honest_acc:.4f}\n")

# ====================================================================
# Step 2: The Trap — Deliberate Leakage of Future Outcome
# ====================================================================
leaky_features = honest_features + ['post_gsc_impressions']
X_leaky = feature_frame[leaky_features].copy()
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)

rf_leaky = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
rf_leaky.fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, rf_leaky.predict_proba(X_te_l)[:, 1])
leaky_acc = accuracy_score(y_te_l, rf_leaky.predict(X_te_l))

print("=== 2. THE TRAP: LEAKY MODEL (with post_gsc_impressions included) ===")
print(f"Features used: {leaky_features}")
print(f"Leaky ROC-AUC : {leaky_auc:.4f}  <-- Score jumps by +{leaky_auc - honest_auc:.4f}!")
print(f"Leaky Accuracy: {leaky_acc:.4f}\n")

# ====================================================================
# Step 3: The Fix — Remove Leaky Column and Retain Honest Features
# ====================================================================
feature_frame.drop(columns=['post_gsc_impressions'], inplace=True)
final_feature_cols = [col for col in feature_frame.columns if col in honest_features]

print("=== 3. THE FIX: REMOVE LEAKY COLUMN ===")
print("Deleted 'post_gsc_impressions' from feature set.")
print(f"Clean features retained: {final_feature_cols}")
print(f"Honest baseline preserved: ROC-AUC = {honest_auc:.4f}, Accuracy = {honest_acc:.4f}")

=== 1. HONEST MODEL (5 knowable pre-decision features) ===
Features used: ['pre_gsc_impressions', 'pre_gsc_clicks', 'pre_avg_position', 'pre_ga4_engaged_sessions', 'content_word_count']
Honest ROC-AUC : 0.6565
Honest Accuracy: 0.7091

=== 2. THE TRAP: LEAKY MODEL (with post_gsc_impressions included) ===
Features used: ['pre_gsc_impressions', 'pre_gsc_clicks', 'pre_avg_position', 'pre_ga4_engaged_sessions', 'content_word_count', 'post_gsc_impressions']
Leaky ROC-AUC : 0.9279  <-- Score jumps by +0.2714!
Leaky Accuracy: 0.7951

=== 3. THE FIX: REMOVE LEAKY COLUMN ===
Deleted 'post_gsc_impressions' from feature set.
Clean features retained: ['pre_gsc_impressions', 'pre_gsc_clicks', 'pre_avg_position', 'pre_ga4_engaged_sessions', 'content_word_count']
Honest baseline preserved: ROC-AUC = 0.6565, Accuracy = 0.7091


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named Limitations of Our Slice:
1. **Severe GA4 Cross-Availability Deficit:** In `month=2026-03`, while Google Search Console data is available for 3.61M rows (36.69%), Google Analytics 4 is available (`IS TRUE`) for only 364,347 rows (3.70% of total rows). Several major clients have hundreds of thousands of daily records with `ga4_data_available = FALSE` (zero-filled). Any model that mandates GA4 engagement features will either discard 96.3% of content-item days or mistake uninstrumented tracking for zero user engagement.
2. **Unbalanced Client Onboarding Depth:** As documented in `dim_clients`, clients have radically different `gsc_data_start` and `ga4_data_start` dates (some start in early 2025, others late 2025 or 2026). A fixed calendar window inevitably samples mature websites with rich baselines alongside newly onboarded sites with thin or missing history.
3. **Partition Edge and Lookahead Horizon:** Iterating inside a single monthly partition (`month=2026-03`) restricts the evaluation horizon to 15 days. Real-world content refresh impacts often take 30–90 days to materialize. Expanding the outcome window across multiple monthly partitions requires careful temporal alignment to avoid overlapping with sealed test partitions (such as `month=2026-06`).

In [6]:
# Prove Limitation 1 & 2: Client-level availability and data start variation
q_limits = f"""
SELECT 
    c.client_hash_id,
    c.is_active,
    c.gsc_data_start,
    c.ga4_data_start,
    COUNT(p.report_date) AS march_total_records,
    COUNT(p.report_date) FILTER (WHERE p.gsc_data_available IS TRUE) AS march_gsc_records,
    COUNT(p.report_date) FILTER (WHERE p.ga4_data_available IS TRUE) AS march_ga4_records
FROM '{CLIENTS_PATH}' c
LEFT JOIN '{PERF_PATH}' p ON c.client_hash_id = p.client_hash_id
GROUP BY c.client_hash_id, c.is_active, c.gsc_data_start, c.ga4_data_start
ORDER BY march_total_records DESC
LIMIT 10
"""
df_limits = con.sql(q_limits).df()
print("Top 10 Clients by March 2026 Volume — Note Clients with 0 GA4 Records:")
display(df_limits)

Top 10 Clients by March 2026 Volume — Note Clients with 0 GA4 Records:


,client_hash_id,is_active,gsc_data_start,ga4_data_start,march_total_records,march_gsc_records,march_ga4_records
0,client_625b6439094e23e4,True,2025-07-01,2026-02-19,988497,0,37
1,client_3ffa76342f366962,True,2025-10-11,2026-03-11,904847,43343,4640
2,client_73cda7b4e4f265ea,True,2025-02-11,2026-03-24,869640,725539,38268
3,client_08a6a72ff48e62c0,True,2025-09-24,NaT,851275,359419,0
4,client_62f4a7e64f5e0096,True,2025-06-07,NaT,756660,610971,0
5,client_65de48885f4ef01b,True,2025-06-21,2026-02-19,426307,28713,4170
6,client_23a62021009f63c4,True,2025-09-24,2025-10-29,423613,388204,146493
7,client_ba65e80a1116ae41,False,2025-10-13,2025-11-09,410409,2990,10523
8,client_2b4306c3ed003f01,True,2026-02-19,NaT,375906,4774,0
9,client_fef1a8f436438636,True,2025-03-11,2026-03-06,335379,254118,48001


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.